<a href="https://colab.research.google.com/github/jaqchien-dotcom/Jacob1-TW/blob/main/03_%E9%9D%9E%E7%B5%90%E6%A7%8B%E5%8C%96%E6%95%B8%E6%93%9A%E6%8E%A1%E9%9B%86.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 非結構化數據採集

黃彬華編撰

## 網路爬蟲概論

網路爬蟲（Web Crawler）是一種自動化程式，專門用來模擬人類瀏覽網頁的行為，自動發送請求並抓取網站上的特定資訊。在進行爬蟲開發時，主要會根據目標網頁的載入機制（靜態或動態網頁），選擇使用靜態或動態策略

| <font size="4">比較項目</font> | <font size="4">靜態策略</font> | <font size="4">動態策略</font> |
| :--- | :--- | :--- |
| <font size="4">核心技術</font> | <font size="4">透過 HTTP 請求套件（如 Python Requests）直接抓取原始碼。</font> | <font size="4">驅動真實瀏覽器（透過 WebDriver）來渲染並操作網頁。</font> |
| <font size="4">適用對象</font> | <font size="4">靜態網頁：資料直接寫死在 HTML 原始碼中。</font> | <font size="4">動態網頁：資料靠 JavaScript、AJAX 或使用者互動後才載入。</font> |
| <font size="4">執行速度</font> | <font size="4">極快。不需下載圖片、不需渲染網頁，只下載純文字檔。</font> | <font size="4">較慢。必須等待瀏覽器開啟、載入元件與執行腳本。</font> |
| <font size="4">資源消耗</font> | <font size="4">極低。對記憶體與 CPU 的負擔非常小。</font> | <font size="4">極高。開啟瀏覽器會佔用大量的系統記憶體與運算資源。</font> |
| <font size="4">常見搭配</font> | <font size="4">搭配 BeautifulSoup 解析 HTML。</font> | <font size="4">搭配 Selenium 操控瀏覽器並取得指定內容。</font> |


## 爬蟲基本技巧

### 先觀察，再爬取

*先觀察*

使用 Chrome 開發人員工具觀察指定區域 (例如：一篇文章範圍)
對指定區域右鍵 > 檢查，會開啟 Elements 頁籤內容供觀察。

*再爬取資料：先爬一，再爬多*

用 select_one() 或 find() 查看單筆資料
OK 後再使用 select() 或 find_all() 取得符合條件的所有資料

下方範例先爬一篇文章測試

In [ ]:
import requests
from bs4 import BeautifulSoup


def one(url):
    print("先爬一篇文章測試")
    try:
        # 發送網路請求
        response = requests.get(url)
        # 如果 status_code 不是 2xx 成功代碼，會在此拋出 HTTPError 異常
        response.raise_for_status()
    except requests.exceptions.HTTPError as e:
        # 捕捉並處理 HTTP 狀態碼錯誤
        print(f"請求失敗，錯誤訊息: {e}")
        return
    except requests.exceptions.RequestException as e:
        # 捕捉其他潛在的網路錯誤（如連線超時、DNS 解析失敗等）
        print(f"網路連線發生非預期錯誤: {e}")
        return

    # 只有在請求成功時才會執行到這裡
    # 建立BeautifulSoup物件
    soup = BeautifulSoup(response.text, "html.parser")
    # 先使用Chrome觀察一篇文章範圍，並用select_one()測試
    article = soup.select_one("div.r-ent")
    # print(f"article:\n{article}")
    # 取出標題
    title = article.select_one(".title").text.strip()
    # print(f"title: {title}")
    # 取出連結
    # 當標題為「本文已被刪除」則<a>為空值
    a = article.select_one(".title > a")
    # print(f"a: {a}")
    link = a["href"] if a else None
    # 取出作者
    author = article.select_one(".author").text.strip()
    # 取出日期
    date = article.select_one(".date").text.strip()
    print("-" * 30)
    print(f"{title}\t{link}\n{author}\t{date}")


def main():
    url = "https://www.ptt.cc/bbs/MobileComm/index.html"
    one(url)


main()


先爬一篇文章測試
------------------------------
[討論] 小米手環10 NFC 悠遊卡請教	/bbs/MobileComm/M.1782989791.A.6C0.html
dntm	7/02


下方範例展示爬取多篇文章

In [ ]:
def many(url):
    print("再爬取多篇文章")
    try:
        # 發送網路請求
        response = requests.get(url)
        # 如果 status_code 不是 2xx 成功代碼，會在此拋出 HTTPError 異常
        response.raise_for_status()
    except requests.exceptions.HTTPError as e:
        # 捕捉並處理 HTTP 狀態碼錯誤
        print(f"請求失敗，錯誤訊息: {e}")
        return
    except requests.exceptions.RequestException as e:
        # 捕捉其他潛在的網路錯誤（如連線超時、DNS 解析失敗等）
        print(f"網路連線發生非預期錯誤: {e}")
        return

    # 建立BeautifulSoup物件
    soup = BeautifulSoup(response.text, "html.parser")
    # 已經觀察過後，改用select()取得符合條件的所有資料
    for article in soup.select("div.r-ent"):
        # 取出標題
        title = article.select_one(".title").text.strip()
        # 取出連結
        # 當標題為「本文已被刪除」則<a>為空值
        a = article.select_one(".title > a")
        link = a["href"] if a else None
        # 取出作者
        author = article.select_one(".author").text.strip()
        # 取出日期
        date = article.select_one(".date").text.strip()
        print("-" * 30)
        print(f"{title}\t{link}\n{author}\t{date}")


def main():
    url = "https://www.ptt.cc/bbs/MobileComm/index.html"
    many(url)


main()

再爬取多篇文章
------------------------------
[討論] 小米手環10 NFC 悠遊卡請教	/bbs/MobileComm/M.1782989791.A.6C0.html
dntm	7/02
------------------------------
[問題] poco x7 pro照片提取文字功能消失	/bbs/MobileComm/M.1782990139.A.C71.html
s2678132	7/02
------------------------------
[置底] 閒聊+代碼分享區	/bbs/MobileComm/M.1721266183.A.DEA.html
starskyjth	7/18
------------------------------
[公告] MobileComm 板規	/bbs/MobileComm/M.1740232019.A.0B8.html
starskyjth	2/22


### Header 偽裝

有些防爬蟲網站會檢查 request headers 甚至 cookies；
需要視情況偽裝 header，但不建議猜測或手寫這些 headers。

*推薦做法*

* 打開 Chrome 瀏覽器，按 F12 進入開發人員工具 > Network 頁籤 > 對請求頁面右鍵選「Copy as cURL」> 打開 https://curlconverter.com/ 線上轉換工具並貼上複製的內容
* 完整複製 headers 到爬蟲程式

*爬蟲程式可依需要加上 headers 與 cookies，測試防爬蟲網站底線的順序為：*

* headers, cookies 都不加
* 只加上 headers
* headers, cookies 都加上

提醒：手動複製的 cookies 通常有效期限不長；一旦過期，爬蟲就會失效

In [ ]:
import requests
from bs4 import BeautifulSoup


def main():
    url = "https://www.ptt.cc/bbs/MobileComm/index.html"

    # 有些防爬蟲網站會檢查 request headers 甚至 cookies
    # Chrome瀏覽器，按 F12 進入開發人員工具 > Network 頁籤 > 對請求頁面右鍵選「Copy as cURL」
    # 打開 https://curlconverter.com/ 線上轉換工具並貼上複製的內容
    # 測試順序：「不加 headers, cookies」-> 「加上 headers」 -> 「加上 headers, cookies」
    headers = {
        'accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7',
        'accept-language': 'zh-TW,zh;q=0.9,en-US;q=0.8,en;q=0.7',
        'cache-control': 'max-age=0',
        'priority': 'u=0, i',
        'sec-ch-ua': '"Google Chrome";v="149", "Chromium";v="149", "Not)A;Brand";v="24"',
        'sec-ch-ua-mobile': '?0',
        'sec-ch-ua-platform': '"macOS"',
        'sec-fetch-dest': 'document',
        'sec-fetch-mode': 'navigate',
        'sec-fetch-site': 'none',
        'sec-fetch-user': '?1',
        'upgrade-insecure-requests': '1',
        'user-agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/149.0.0.0 Safari/537.36',
    }

    # 提醒：手動複製的 cookies 通常都有有效期限。少則幾十分鐘，多則幾天。一旦過期，爬蟲就會再度失效
    cookies = {
        '_gid': 'GA1.2.584770886.1781837058',
        '_gat': '1',
        '_ga': 'GA1.1.2058548064.1780645983',
        '_ga_DZ6Y3BY9GW': 'GS2.1.s1781837058$o3$g1$t1781837618$j60$l0$h0',
    }

    try:
        # 發送網路請求
        response = requests.get(url, headers=headers, cookies=cookies)
        # 如果 status_code 不是 2xx 成功代碼，會在此拋出 HTTPError 異常
        response.raise_for_status()
    except requests.exceptions.HTTPError as e:
        # 捕捉並處理 HTTP 狀態碼錯誤
        print(f"請求失敗，錯誤訊息: {e}")
        return
    except requests.exceptions.RequestException as e:
        # 捕捉其他潛在的網路錯誤（如連線超時、DNS 解析失敗等）
        print(f"網路連線發生非預期錯誤: {e}")
        return

    print("URL:", url)
    # 建立BeautifulSoup物件
    soup = BeautifulSoup(response.text, "html.parser")
    for article in soup.select("div.r-ent"):
        print("-" * 30)
        # 取出標題
        title = article.select_one(".title").text.strip()
        # 取出連結
        # 當標題為「本文已被刪除」則<a>為空值
        a = article.select_one(".title > a")
        link = a["href"] if a else None
        # 取出作者
        author = article.select_one(".author").text.strip()
        # 取出日期
        date = article.select_one(".date").text.strip()
        print(f"{title}\t{link}\n{author}\t{date}")


main()

URL: https://www.ptt.cc/bbs/MobileComm/index.html
------------------------------
[討論] 小米手環10 NFC 悠遊卡請教	/bbs/MobileComm/M.1782989791.A.6C0.html
dntm	7/02
------------------------------
[問題] poco x7 pro照片提取文字功能消失	/bbs/MobileComm/M.1782990139.A.C71.html
s2678132	7/02
------------------------------
[置底] 閒聊+代碼分享區	/bbs/MobileComm/M.1721266183.A.DEA.html
starskyjth	7/18
------------------------------
[公告] MobileComm 板規	/bbs/MobileComm/M.1740232019.A.0B8.html
starskyjth	2/22


### 進一步優化

* 加入自動重試機制，避免偶發性斷線
* 加上 timeout 逾時設定，防止程式因伺服器沒反應而卡死
* 改寫為 Session 模式以優化多次請求的效率

In [ ]:
import requests
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from urllib3.util import Retry


# 優化爬蟲程式
def fetch_page(url: str, headers: dict, cookies: dict):
    # 建立 Session 物件（優化連線效率，會自動維持 headers 與 cookies 狀態）
    session = requests.Session()
    session.headers.update(headers)
    session.cookies.update(cookies)

    # 定義自動重試機制
    retries = Retry(
        # 最多重試 3 次
        total=3,
        # Retry採用的是指數退避演算法：backoff_factor * (2 ** (retry_count - 1))
        # backoff_factor=1，第一次重試時retry_count=1，即「1 * (2 ** (1 - 1)) = 1秒」, 其後重試為 2秒, 4秒
        backoff_factor=1,
        # 只有在遇到網路斷線、或伺服器崩潰的狀態碼（500, 502, 503, 504）時才會觸發重試
        # 網頁不存在（404）或被封鎖（403）即使重試 100 次結果也一樣，直接跳出才不會浪費資源與時間
        status_forcelist=[500, 502, 503, 504],
        # 讓錯誤狀態碼留給 raise_for_status() 處理，以便精確捕捉 HTTPError
        raise_on_status=False,
    )

    # 將重試機制掛載到 HTTP 與 HTTPS 協定上
    adapter = HTTPAdapter(max_retries=retries)
    session.mount("http://", adapter)
    session.mount("https://", adapter)

    try:
        # 使用 session 發送網路請求，並加上 timeout 逾時設定（防程式卡死）
        # timeout=(連線逾時秒數, 回傳資料逾時秒數)
        response = session.get(url, timeout=(5, 10))

        # 如果 status_code 不是 2xx 成功代碼，會在此拋出 HTTPError 異常
        response.raise_for_status()

        # 成功，回傳回應物件
        return response
    except requests.exceptions.Timeout as e:
        # 專門捕捉超時錯誤
        print(f"請求逾時，伺服器可能因負載過重未回應: {e}")
    except requests.exceptions.HTTPError as e:
        # 捕捉並處理 HTTP 狀態碼錯誤（在自動重試 3 次都失敗後，才會走到這裡）
        print(f"請求失敗，錯誤狀態碼: {e}")
    except requests.exceptions.RequestException as e:
        # 捕捉其他潛在的網路底層錯誤
        print(f"網路連線發生非預期錯誤: {e}")
    finally:
        # 良好習慣：關閉 session 釋放連線資源
        session.close()

    # 發生異常，回傳 None
    return None


def parse_html(html_text: str):
    print("Article List:")
    # 建立BeautifulSoup物件
    soup = BeautifulSoup(html_text, "html.parser")
    for article in soup.select("div.r-ent"):
        print("-" * 30)
        # 取出標題
        title = article.select_one(".title").text.strip()
        # 取出連結
        # 當標題為「本文已被刪除」則<a>為空值
        a = article.select_one(".title > a")
        link = a["href"] if a else None
        # 取出作者
        author = article.select_one(".author").text.strip()
        # 取出日期
        date = article.select_one(".date").text.strip()
        print(f"{title}\t{link}\n{author}\t{date}")


def main():
    url = "https://www.ptt.cc/bbs/MobileComm/index.html"

    headers = {
        "accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7",
        "accept-language": "zh-TW,zh;q=0.9,en-US;q=0.8,en;q=0.7",
        "cache-control": "max-age=0",
        "priority": "u=0, i",
        "sec-ch-ua": '"Google Chrome";v="149", "Chromium";v="149", "Not)A;Brand";v="24"',
        "sec-ch-ua-mobile": "?0",
        "sec-ch-ua-platform": '"macOS"',
        "sec-fetch-dest": "document",
        "sec-fetch-mode": "navigate",
        "sec-fetch-site": "none",
        "sec-fetch-user": "?1",
        "upgrade-insecure-requests": "1",
        "user-agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/149.0.0.0 Safari/537.36",
    }

    cookies = {
        "_gid": "GA1.2.584770886.1781837058",
        "_gat": "1",
        "_ga": "GA1.1.2058548064.1780645983",
        "_ga_DZ6Y3BY9GW": "GS2.1.s1781837058$o3$g1$t1781837618$j60$l0$h0",
    }

    # 取得網頁回應
    response = fetch_page(url, headers, cookies)

    # 判斷是否有成功取得回應，成功才丟給解析函式
    if response:
        parse_html(response.text)


main()

Article List:
------------------------------
[討論] 小米手環10 NFC 悠遊卡請教	/bbs/MobileComm/M.1782989791.A.6C0.html
dntm	7/02
------------------------------
[問題] poco x7 pro照片提取文字功能消失	/bbs/MobileComm/M.1782990139.A.C71.html
s2678132	7/02
------------------------------
[置底] 閒聊+代碼分享區	/bbs/MobileComm/M.1721266183.A.DEA.html
starskyjth	7/18
------------------------------
[公告] MobileComm 板規	/bbs/MobileComm/M.1740232019.A.0B8.html
starskyjth	2/22


## 爬多頁內容

只爬單一頁內容的資料量通常不足以進行資料探勘、統計分析或機器學習訓練，所以在實際應用與數據分析中，通常需要爬取多頁，也就是大量資料。

*觀察分頁邏輯*

要爬取多頁資料，第一步就是使用 Chrome 開發人員工具觀察去上、下頁的超連結 (分頁邏輯)。

*隨機延遲機制*

爬多頁時，因為電腦執行速度快，爬蟲程式數秒內就會對目標網站發出 n 個請求，可能會被該網站認為是惡意程式，而導致被伺服器暫時封鎖 IP。

要避免上述問題最簡單方式就是加上隨機延遲 (random delay) 機制，也就是在每頁爬取之間隨機暫停 1 到 2 秒，可以讓爬蟲行為更像真人瀏覽。

In [ ]:
import time
import random
import requests
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from urllib3.util import Retry


# 優化爬蟲程式
def fetch_page(url: str, headers: dict, cookies: dict):
    # 建立 Session 物件（優化連線效率，會自動維持 headers 與 cookies 狀態）
    session = requests.Session()
    session.headers.update(headers)
    session.cookies.update(cookies)

    # 定義自動重試機制
    retries = Retry(
        # 最多重試 3 次
        total=3,
        # Retry採用的是指數退避演算法：backoff_factor * (2 ** (retry_count - 1))
        # backoff_factor=1，第一次重試時retry_count=1，即「1 * (2 ** (1 - 1)) = 1秒」, 其後重試為 2秒, 4秒
        backoff_factor=1,
        # 只有在遇到網路斷線、或伺服器崩潰的狀態碼（500, 502, 503, 504）時才會觸發重試
        # 網頁不存在（404）或被封鎖（403）即使重試 100 次結果也一樣，直接跳出才不會浪費資源與時間
        status_forcelist=[500, 502, 503, 504],
        # 讓錯誤狀態碼留給 raise_for_status() 處理，以便精確捕捉 HTTPError
        raise_on_status=False,
    )

    # 將重試機制掛載到 HTTP 與 HTTPS 協定上
    adapter = HTTPAdapter(max_retries=retries)
    session.mount("http://", adapter)
    session.mount("https://", adapter)

    try:
        # 使用 session 發送網路請求，並加上 timeout 逾時設定（防程式卡死）
        # timeout=(連線逾時秒數, 回傳資料逾時秒數)
        response = session.get(url, timeout=(5, 10))

        # 如果 status_code 不是 2xx 成功代碼，會在此拋出 HTTPError 異常
        response.raise_for_status()

        # 成功，回傳回應物件
        return response
    except requests.exceptions.Timeout as e:
        # 專門捕捉超時錯誤
        print(f"請求逾時，伺服器可能因負載過重未回應: {e}")
    except requests.exceptions.HTTPError as e:
        # 捕捉並處理 HTTP 狀態碼錯誤（在自動重試 3 次都失敗後，才會走到這裡）
        print(f"請求失敗，錯誤狀態碼: {e}")
    except requests.exceptions.RequestException as e:
        # 捕捉其他潛在的網路底層錯誤
        print(f"網路連線發生非預期錯誤: {e}")
    finally:
        # 良好習慣：關閉 session 釋放連線資源
        session.close()

    # 發生異常，回傳 None
    return None


def parse_html(html_text: str):
    # 建立BeautifulSoup物件
    soup = BeautifulSoup(html_text, "html.parser")
    for article in soup.select("div.r-ent"):
        print("-" * 30)
        # 取出標題
        title = article.select_one(".title").text.strip()
        # 取出連結
        # 當標題為「本文已被刪除」則<a>為空值
        a = article.select_one(".title > a")
        link = a["href"] if a else None
        # 取出作者
        author = article.select_one(".author").text.strip()
        # 取出日期
        date = article.select_one(".date").text.strip()
        print(f"{title}\t{link}\n{author}\t{date}")


# 從網頁中解析出「上頁」的網址
def get_prev_url(html_text: str) -> str:
    soup = BeautifulSoup(html_text, "html.parser")
    # 尋找含有「‹ 上頁」文字的按鈕
    prev_btn = soup.find("a", string="‹ 上頁")
    if prev_btn and prev_btn.get("href"):
        return "https://www.ptt.cc" + prev_btn["href"]
    return None


def main():
    url = "https://www.ptt.cc/bbs/MobileComm/index.html"
    # 欲取得較完整 headers 參看 Basic02Demo
    headers = {}
    cookies = {}

    # 讓使用者輸入欲爬取的頁數（並含防呆機制）
    while True:
        try:
            user_input = input("請輸入欲爬取的總頁數：")
            total_pages_to_crawl = int(user_input)
            if total_pages_to_crawl > 0:
                break
            else:
                print("請輸入大於 0 的正整數。")
        except ValueError:
            print("輸入格式錯誤！請輸入有效的數字。")

    print(f"\n開始爬取，目標頁數：{total_pages_to_crawl} 頁")

    for page_num in range(1, total_pages_to_crawl + 1):
        if not url:
            print("找不到下一頁的網址，停止爬取。")
            break

        print(f"\n{'='*10} 正在爬取第 {page_num} 頁 {'='*10}")
        print(f"網址: {url}")

        # 取得網頁回應
        response = fetch_page(url, headers, cookies)

        # 判斷是否有成功取得回應，成功才丟給解析函式
        if response:
            parse_html(response.text)

            # 更新 url 為「上一頁」的網址，供下一次迴圈使用
            url = get_prev_url(response.text)

            # 如果還沒爬完，且還有下一頁，就執行隨機延遲
            if page_num < total_pages_to_crawl and url:
                sleep_time = random.uniform(1.0, 2.0)
                print("\n" + "=" * 30)
                print(f"等待 {sleep_time:.2f} 秒後繼續下一頁...")
                print("=" * 30)
                time.sleep(sleep_time)
        else:
            print(f"第 {page_num} 頁請求失敗，終止後續頁面爬取。")
            break


main()

請輸入欲爬取的總頁數：3

開始爬取，目標頁數：3 頁

========== 正在爬取第 1 頁 ==========
網址: https://www.ptt.cc/bbs/MobileComm/index.html
------------------------------
[討論] 小米手環10 NFC 悠遊卡請教	/bbs/MobileComm/M.1782989791.A.6C0.html
dntm	7/02
------------------------------
[問題] poco x7 pro照片提取文字功能消失	/bbs/MobileComm/M.1782990139.A.C71.html
s2678132	7/02
------------------------------
[置底] 閒聊+代碼分享區	/bbs/MobileComm/M.1721266183.A.DEA.html
starskyjth	7/18
------------------------------
[公告] MobileComm 板規	/bbs/MobileComm/M.1740232019.A.0B8.html
starskyjth	2/22

等待 1.61 秒後繼續下一頁...

========== 正在爬取第 2 頁 ==========
網址: https://www.ptt.cc/bbs/MobileComm/index7646.html
------------------------------
[方案] 中華/遠傳 4G吃到飽方案請益	/bbs/MobileComm/M.1782810734.A.B37.html
michael4210	6/30
------------------------------
Re: [心得] 如果用pixel手機的，去日本不建議用esim	/bbs/MobileComm/M.1782873643.A.84F.html
ls4860	7/01
------------------------------
[情報] OPPO Reno16系列台灣價格在遠傳搶先曝光	/bbs/MobileComm/M.1782873799.A.B26.html
betweenus020	7/01
------------------

## 爬蟲遇到 Cookies

先使用瀏覽器拜訪 PTT 八卦版 (https://www.ptt.cc/bbs/Gossiping/index.html)

開啟 Chrome 開發人員工具，觀察點擊「我同意，我已年滿十八歲」按鈕的結果：

* Network > Cookies 記錄 Response Cookies 有 "over18=1" 資訊

* 之後再次進入八卦版，Request Cookies 就會帶有 "over18=1" 資訊，也不再有要求同意的畫面

In [ ]:
import requests
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from urllib3.util import Retry


# 優化爬蟲程式：負責處理安全的網路請求
def fetch_page(url: str, headers: dict, cookies: dict):
    # 建立 Session 物件（優化連線效率，會自動維持 headers 與 cookies 狀態）
    session = requests.Session()
    session.headers.update(headers)
    session.cookies.update(cookies)

    # 定義自動重試機制
    retries = Retry(
        # 最多重試 3 次
        total=3,
        # Retry採用的是指數退避演算法：backoff_factor * (2 ** (retry_count - 1))
        # backoff_factor=1，第一次重試時retry_count=1，即「1 * (2 ** (1 - 1)) = 1秒」, 其後重試為 2秒, 4秒
        backoff_factor=1,
        # 只有在遇到網路斷線、或伺服器崩潰的狀態碼（500, 502, 503, 504）時才會觸發重試
        # 網頁不存在（404）或被封鎖（403）即使重試 100 次結果也一樣，直接跳出才不會浪費資源與時間
        status_forcelist=[500, 502, 503, 504],
        # 讓錯誤狀態碼留給 raise_for_status() 處理，以便精確捕捉 HTTPError
        raise_on_status=False,
    )

    # 將重試機制掛載到 HTTP 與 HTTPS 協定上
    adapter = HTTPAdapter(max_retries=retries)
    session.mount("http://", adapter)
    session.mount("https://", adapter)

    try:
        # 使用 session 發送網路請求，並加上 timeout 逾時設定（防程式卡死）
        # timeout=(連線逾時秒數, 回傳資料逾時秒數)
        response = session.get(url, timeout=(5, 10))

        # 如果 status_code 不是 2xx 成功代碼，會在此拋出 HTTPError 異常
        response.raise_for_status()

        # 成功，回傳回應物件
        return response
    except requests.exceptions.Timeout as e:
        # 專門捕捉超時錯誤
        print(f"請求逾時，伺服器可能因負載過重未回應: {e}")
    except requests.exceptions.HTTPError as e:
        # 捕捉並處理 HTTP 狀態碼錯誤（在自動重試 3 次都失敗後，才會走到這裡）
        print(f"請求失敗，錯誤狀態碼: {e}")
    except requests.exceptions.RequestException as e:
        # 捕捉其他潛在的網路底層錯誤
        print(f"網路連線發生非預期錯誤: {e}")
    finally:
        # 良好習慣：關閉 session 釋放連線資源
        session.close()

    # 發生異常，回傳 None
    return None


def parse_html(html_text: str):
    print("Article List:")
    # 建立BeautifulSoup物件
    soup = BeautifulSoup(html_text, "html.parser")
    for article in soup.select("div.r-ent"):
        print("-" * 30)
        # 取出標題
        title = article.select_one(".title").text.strip()
        # 取出連結
        # 當標題為「本文已被刪除」則<a>為空值
        a = article.select_one(".title > a")
        link = a["href"] if a else None
        # 取出作者
        author = article.select_one(".author").text.strip()
        # 取出日期
        date = article.select_one(".date").text.strip()
        print(f"{title}\t{link}\n{author}\t{date}")


def main():
    # 目標頁面
    target_url = "https://www.ptt.cc/bbs/Gossiping/index.html"

    # 欲取得較完整 headers 參看 Basic02Demo
    headers = {
        "user-agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/96.0.4664.93 Safari/537.36"
    }

    # 開啟 Chrome 開發人員工具，觀察點擊「我同意，我已年滿十八歲」的結果
    # Network > Cookies 記錄 Response Cookies 有 "over18=1" 資訊
    # 之後再次進入八卦版，Request Cookies 就會帶有 "over18=1" 資訊，也不再有要求同意的畫面
    # 帶入「我同意...」的 Cookie，直接跳過 POST 點擊「我同意...」按鈕的步驟
    cookies = {"over18": "1"}

    response = fetch_page(target_url, headers=headers, cookies=cookies)

    # 判斷是否有成功取得回應，成功才丟給解析函式
    if response:
        parse_html(response.text)


main()

Article List:
------------------------------
[問卦] 底層的生活是不是真的需要點小確幸？	/bbs/Gossiping/M.1782999843.A.07F.html
itachi6060	7/02
------------------------------
[問卦] 明天開盤all in大統益 能退休嗎？	/bbs/Gossiping/M.1782999916.A.1E0.html
THEHELY	7/02
------------------------------
[問卦] 脆熱議：16歲就當通緝犯！	/bbs/Gossiping/M.1783000046.A.D89.html
pigpigcom123	7/02
------------------------------
Re: [新聞] 獨家／癌逝女躺醫院10多天沒人接走…父	/bbs/Gossiping/M.1783000131.A.94D.html
arnold3	7/02
------------------------------
[問卦] 阿財客家發粿突然爆紅？	/bbs/Gossiping/M.1783000396.A.D1C.html
YingWenTsai	7/02
------------------------------
[新聞] 快訊／「銅鑼灣書店」老闆林榮基癌症病逝	/bbs/Gossiping/M.1783000441.A.4EF.html
proprome	7/02
------------------------------
[問卦] 七月半團長說話了：把選擇權還給孩童	/bbs/Gossiping/M.1783000455.A.D5D.html
psychohero	7/02
------------------------------
[問卦] Kia還要在台灣賣車嗎？	/bbs/Gossiping/M.1783000503.A.ED0.html
pptsuck	7/02
------------------------------
[新聞] 有錢不能自由運用？軍公教加薪4% 基層公	/bbs/Gossiping/M.1783000517.A.54F.html
abiann	7/02
-------------------

## Selenium

* Selenium 是一個自動化測試工具，可用來操縱瀏覽器
* 可以模仿真人操作網頁的各種行為 (例如：點擊按鈕、輸入資料)
* 因為擬真度很高，目標網站不易阻擋；因為如果阻擋意味著也很可能將真人操作阻擋在外
* 可以解決前述 Cookies 問題或是安全憑證問題

### Selenium 套件安裝

In [ ]:
# 安裝 selenium，-U 表示升級到最新版
!pip install -U selenium

In [ ]:
# Colab 需自行下載並安裝 Chrome
# 更新套件清單
!apt-get update
# 下載 Google Chrome stable .deb 安裝檔
!wget https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
# 使用 dpkg 安裝 .deb 檔
!dpkg -i google-chrome-stable_current_amd64.deb
# 解決可能存在的依賴問題
!apt --fix-broken install

# 清理下載的 .deb 檔
!rm google-chrome-stable_current_amd64.deb

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://cli.github.com/packages stable InRelease [3,917 B]
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:9 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,080 kB]
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,326 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,308 kB]
Get:13 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packag

In [ ]:
# 查詢 chrome 版本以確定是否安裝成功
!google-chrome --version

Google Chrome 150.0.7871.46 


可使用 Selenium 模擬點擊「我同意，我已年滿十八歲」按鈕解決之前 PTT 八卦版 Cookies 問題

In [ ]:
import time
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


def parse_html(html_text: str):
    # 建立BeautifulSoup物件
    soup = BeautifulSoup(html_text, "html.parser")
    for article in soup.select("div.r-ent"):
        print("-" * 30)
        # 取出標題
        title = article.select_one(".title").text.strip()
        # 取出連結
        # 當標題為「本文已被刪除」則<a>為空值
        a = article.select_one(".title > a")
        link = a["href"] if a else None
        # 取出作者
        author = article.select_one(".author").text.strip()
        # 取出日期
        date = article.select_one(".date").text.strip()
        print(f"{title}\t{link}\n{author}\t{date}")


def main():
    # 建立瀏覽器設定物件
    options = webdriver.ChromeOptions()

    # 啟用無頭模式，並加入其他 Colab 環境所需的參數
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    # 明確指定 Chrome 執行檔的路徑
    options.binary_location = "/usr/bin/google-chrome"

    # 降低被偵測為自動化機率
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option('useAutomationExtension', False)

    # 建立 Chrome Driver (Selenium 4.x 以上最推薦的簡潔寫法，會自動管理驅動)
    driver = webdriver.Chrome(options=options)

    try:
        url = "https://www.ptt.cc/bbs/Gossiping/index.html"
        driver.get(url)

        # 優化：使用顯式等待 (Explicit Wait) 替代 time.sleep(1)
        # 最多等待 5 秒，直到 name="yes" 的按鈕可在畫面上被點擊
        wait = WebDriverWait(driver, 5)
        yes_button = wait.until(EC.element_to_be_clickable((By.NAME, "yes")))
        yes_button.click()

        # 確認網頁成功跳轉（等待網址改變或特定元素出現）
        wait.until(EC.url_contains("index.html"))
        print(f"目前網址:\n{driver.current_url}")

        # 取得網頁內容並解析
        parse_html(driver.page_source)

    except Exception as e:
        print(f"自動化操作發生錯誤: {e}")

    finally:
        # 確保不論成功或失敗，瀏覽器都會關閉釋放記憶體
        time.sleep(2)
        driver.quit()


main()

目前網址:
https://www.ptt.cc/bbs/Gossiping/index.html
------------------------------
[問卦] 底層的生活是不是真的需要點小確幸？	/bbs/Gossiping/M.1782999843.A.07F.html
itachi6060	7/02
------------------------------
[問卦] 明天開盤all in大統益 能退休嗎？	/bbs/Gossiping/M.1782999916.A.1E0.html
THEHELY	7/02
------------------------------
[問卦] 脆熱議：16歲就當通緝犯！	/bbs/Gossiping/M.1783000046.A.D89.html
pigpigcom123	7/02
------------------------------
Re: [新聞] 獨家／癌逝女躺醫院10多天沒人接走…父	/bbs/Gossiping/M.1783000131.A.94D.html
arnold3	7/02
------------------------------
[問卦] 阿財客家發粿突然爆紅？	/bbs/Gossiping/M.1783000396.A.D1C.html
YingWenTsai	7/02
------------------------------
[新聞] 快訊／「銅鑼灣書店」老闆林榮基癌症病逝	/bbs/Gossiping/M.1783000441.A.4EF.html
proprome	7/02
------------------------------
[問卦] 七月半團長說話了：把選擇權還給孩童	/bbs/Gossiping/M.1783000455.A.D5D.html
psychohero	7/02
------------------------------
[問卦] Kia還要在台灣賣車嗎？	/bbs/Gossiping/M.1783000503.A.ED0.html
pptsuck	7/02
------------------------------
[新聞] 有錢不能自由運用？軍公教加薪4% 基層公	/bbs/Gossiping/M.1783000517.A.54F.

下面範例爬取司法院判決書：重度動態網頁、有 iframe、分頁完全依賴 JS，最好仰賴 Selenium

In [ ]:
import time
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


def crawl_judgment(keyword):
    # 建立瀏覽器設定物件
    options = webdriver.ChromeOptions()

    # 啟用無頭模式，並加入其他 Colab 環境所需的參數
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    # 明確指定 Chrome 執行檔的路徑
    options.binary_location = "/usr/bin/google-chrome"

    # 降低被偵測為自動化機率
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option('useAutomationExtension', False)

    # 建立 Chrome Driver (Selenium 4.x 以上最新簡潔寫法，免手動丟 Service)
    driver = webdriver.Chrome(options=options)

    # 設定顯式等待工具，最多等待 10 秒
    wait = WebDriverWait(driver, 10)

    try:
        # 開啟網頁
        driver.get("https://judgment.judicial.gov.tw/FJUD/default.aspx")

        # 找到輸入框並輸入關鍵字
        search_input = driver.find_element(By.NAME, "txtKW")
        search_input.clear()
        search_input.send_keys(keyword)

        # 點擊送出按鈕(ID為btnSimpleQry)
        submit_button = driver.find_element(By.ID, "btnSimpleQry")
        submit_button.click()

        # 切換到iframe
        # 會「自動等待」直到 iframe 載入完成並「直接切換進去」
        wait.until(EC.frame_to_be_available_and_switch_to_it((By.ID, "iframe-data")))

        # 確保 iframe 內部的資料表格已經渲染出來並可操作
        wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "table#jud")))

        # 取得iframe裡的HTML
        html = driver.page_source
        soup = BeautifulSoup(html, "html.parser")

        # 存放判決書的表格
        table = soup.select_one("table#jud")

        if not table:
            print("找不到任何裁判書資料，請確認關鍵字是否正確。")
            return

        # 表格標題(第一個tr即為標題列)
        tr = table.tr
        # 所有表格標題
        ths = tr("th")
        for th in ths:# 表格標題 (第一個 tr 即為標題列)
            tr_title = table.tr
            if tr_title:
                ths = tr_title("th")
                for th in ths:
                    print(th.text.strip(), end="\t")
                print()

            trs = table("tr")
            # 第一個 tr 為標題列，跳過
            trs = trs[1:]

            # 每個判決書由 2 個 tr 組成（基礎資料與摘要）
            for tr in trs:
                # 檢查tr是否有「class="summary"」；如果沒有class屬性就回傳[]而非None，
                # 避免「"summary" in None」造成執行錯誤
                # 有「class="summary"」代表判決書摘要
                if "summary" in tr.get("class", []):
                    print("-" * 20)
                    span = tr.span
                    content = span.text.strip() if span else "無摘要內容"
                    print(content)
                else:
                    print("=" * 30)
                    # 判決書的字號、日期、案由
                    tds = tr("td")
                    for td in tds:
                        print(td.text.strip().replace("\n", " "), end="  ")
                    print()

    except Exception as e:
        print(f"自動化操作或解析發生錯誤: {e}")

    finally:
        # 短暫停留在觀看畫面，然後再關閉 Chrome
        time.sleep(2)
        driver.quit()


def main():
    keyword = input("輸入搜尋關鍵字: ")
    crawl_judgment(keyword)


main()

輸入搜尋關鍵字: 竊盜
序號	裁判字號 （內容大小）	裁判日期	裁判案由	
1.  臺灣士林地方法院 114 年度 易 字第 852 號刑事裁定（1K）  115.07.01  竊盜  
--------------------
臺灣士林地方法院刑事裁定114年度易字第852號公訴人臺灣士林地方檢察署檢察官被告劉原復上列被告因竊盜案件，經檢察官提起公訴（114年度偵字第1375號），本院裁定如下：主文本件繼續審判。理由一、按刑事訴訟法第294條第1項、第2項停止審判之原因消滅時，法院應...
2.  臺灣新竹地方法院 115 年度 聲 字第 665 號刑事裁定（4K）  115.07.01  聲請定其應執行刑  
--------------------
...第4976號 (已執畢) 新竹地檢113年度執字第4754號(已執畢) 新竹地檢113年度執字第4869號(已執畢) 編號4 罪名竊盜 宣告刑有期徒刑4月 犯罪日期113/01/17偵查(自訴)機關年度及案號新竹地檢113年度偵字第2851號 最後 事實審 法院 ...
3.  臺灣士林地方法院 115 年度 審簡 字第 714 號刑事判決（5K）  115.07.01  竊盜  
--------------------
...國 115年3 月12日書記官鄭雅文附錄本案所犯法條全文中華民國刑法第320條意圖為自己或第三人不法之所有，而竊取他人之動產者，為竊盜罪，處五年以下有期徒刑、拘役或五十萬元以下罰金。意圖為自己或第三人不法之利益，而竊佔他人之不動產者，依前項之規定處斷。前二項之未遂...
4.  臺灣士林地方法院 115 年度 審簡 字第 701 號刑事判決（5K）  115.07.01  竊盜  
--------------------
... 115年4 月 2日書記官林秀玉附錄本案所犯法條全文：中華民國刑法第320條意圖為自己或第三人不法之所有，而竊取他人之動產者，為竊盜罪，處 5 年以下有期徒刑、拘役或 50 萬元以下罰金。意圖為自己或第三人不法之利益，而竊佔他人之不動產者，依前項之規定處斷。前二...
5.  臺灣士林地方法院 115 年度 簡 字第 127 號刑事判決（6K）  115.07.01  竊盜等  
--------------------
...條第1項第1款入住宅竊盜、同

## 隨機延遲

雖然 Selenium 擬真度很高，目標網站不易阻擋，但程式的點擊、移動、換頁都在精準的 1.00 秒完成，伺服器防火牆就會判定為機器人

可以加入隨機延遲 (Random Delay) 機制，讓點擊行為更像真人、更難被偵測

在 Selenium 爬蟲中加入隨機延遲 (Random Delay) 是一個非常專業的防爬蟲策略

司法院網站（或其他政府機關網站）內可能建有流量監控機制，如果程式每次點擊送出、換頁、解析的間隔時間都是精準的整數（例如剛好 2 秒），伺服器的防火牆很容易判定這是「機器人行為」而將你的 IP 封鎖。加入隨機延遲可以模仿人類點擊時的不規則時間差，讓操作行為看起來更像真人。

我們需要引入 Python 內建的 random 模組，並在關鍵的步驟（如送出查詢、切換分頁前後）加入隨機秒數。

In [ ]:
import random
import time
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


# 建立一個隨機延遲的輔助函式
def random_delay(min_sec=1.5, max_sec=3.5):
    """在設定的秒數區間內，隨機挑選一個浮點數秒數進行等待，模擬真人停頓。"""
    delay = random.uniform(min_sec, max_sec)
    print(f"模擬真人操作，隨機停頓 {delay:.2f} 秒...")
    time.sleep(delay)


def crawl_judgment(keyword):
    options = webdriver.ChromeOptions()

    # 啟用無頭模式，並加入其他 Colab 環境所需的參數
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    # 明確指定 Chrome 執行檔的路徑
    options.binary_location = "/usr/bin/google-chrome"

    # 降低被偵測為自動化機率
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)
    options.add_argument("--start-maximized")

    # 建立 Chrome Driver
    driver = webdriver.Chrome(options=options)

    # 設定顯式等待工具，最多等待 10 秒
    wait = WebDriverWait(driver, 20)

    try:
        print("正在開啟司法院網頁...")
        # 更正為正確的司法院判決書查詢頁面網址
        driver.get("https://judgment.judicial.gov.tw/FJUD/default.aspx")

        # 1. 模擬人類在輸入前的隨機猶豫時間
        random_delay(1.0, 2.5)

        # 確保輸入框可以被操作
        search_input = wait.until(
            EC.element_to_be_clickable((By.NAME, "txtKW"))
        )
        search_input.clear()

        # 2. 模擬真人輸入（這部分維持直接輸入，但在點擊送出前稍作停頓）
        search_input.send_keys(keyword)
        random_delay(1.2, 2.8)

        # 點擊送出按鈕
        submit_button = driver.find_element(By.ID, "btnSimpleQry")
        submit_button.click()
        print(
            f"已送出關鍵字 [{keyword}]，正在等待司法院伺服器回應..."
        )

        # 3. 優先等待 iframe 元素產生並切換進去
        wait.until(
            EC.frame_to_be_available_and_switch_to_it((By.ID, "iframe-data"))
        )

        # 4. 切換進去後，等待結果表格（table#jud）出現在 DOM 中
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "table#jud")))

        # 在擷取資料前，再隨機停頓一下，讓瀏覽器完全渲染完畢
        random_delay(1.5, 3.0)
        print("資料載入成功，開始解析內容...\n")

        # 取得 iframe 裡的 HTML 原始碼並交給 BeautifulSoup 解析
        html = driver.page_source
        soup = BeautifulSoup(html, "html.parser")
        table = soup.select_one("table#jud")

        if not table:
            print("找不到任何裁判書資料，請確認關鍵字是否正確。")
            return

        # 表格標題
        tr_title = table.tr
        if tr_title:
            ths = tr_title("th")
            for th in ths:
                print(th.text.strip(), end="\t")
            print()

        # 處理資料列
        trs = table("tr")[1:]  # 跳過標題
        for tr in trs:
            if "summary" in tr.get("class", []):
                span = tr.span
                content = span.text.strip() if span else "無摘要內容"
                print(content)
                print("-" * 30)
            else:
                tds = tr("td")
                for td in tds:
                    print(
                        td.text.strip().replace("\n", " "), end="  "
                    )
                print()

    except TimeoutException:
        print("請求逾時！司法院伺服器回應過慢，或是該關鍵字查無資料。")
    except Exception as e:
        print(f"發生非預期錯誤: {e}")

    finally:
        # 結束後也隨機留存一下畫面再關閉
        random_delay(2.0, 4.0)
        driver.quit()
        print("\n瀏覽器已安全關閉。")


def main():
    keyword = input("輸入搜尋關鍵字: ")
    if keyword.strip():
        crawl_judgment(keyword)
    else:
        print("關鍵字不能為空！")


main()

輸入搜尋關鍵字: 竊盜
正在開啟司法院網頁...
模擬真人操作，隨機停頓 1.48 秒...
模擬真人操作，隨機停頓 1.58 秒...
已送出關鍵字 [竊盜]，正在等待司法院伺服器回應...
模擬真人操作，隨機停頓 2.81 秒...
資料載入成功，開始解析內容...

序號	裁判字號 （內容大小）	裁判日期	裁判案由	
1.  臺灣士林地方法院 114 年度 易 字第 852 號刑事裁定（1K）  115.07.01  竊盜  
臺灣士林地方法院刑事裁定114年度易字第852號公訴人臺灣士林地方檢察署檢察官被告劉原復上列被告因竊盜案件，經檢察官提起公訴（114年度偵字第1375號），本院裁定如下：主文本件繼續審判。理由一、按刑事訴訟法第294條第1項、第2項停止審判之原因消滅時，法院應...
------------------------------
2.  臺灣新竹地方法院 115 年度 聲 字第 665 號刑事裁定（4K）  115.07.01  聲請定其應執行刑  
...第4976號 (已執畢) 新竹地檢113年度執字第4754號(已執畢) 新竹地檢113年度執字第4869號(已執畢) 編號4 罪名竊盜 宣告刑有期徒刑4月 犯罪日期113/01/17偵查(自訴)機關年度及案號新竹地檢113年度偵字第2851號 最後 事實審 法院 ...
------------------------------
3.  臺灣士林地方法院 115 年度 審簡 字第 714 號刑事判決（5K）  115.07.01  竊盜  
...國 115年3 月12日書記官鄭雅文附錄本案所犯法條全文中華民國刑法第320條意圖為自己或第三人不法之所有，而竊取他人之動產者，為竊盜罪，處五年以下有期徒刑、拘役或五十萬元以下罰金。意圖為自己或第三人不法之利益，而竊佔他人之不動產者，依前項之規定處斷。前二項之未遂...
------------------------------
4.  臺灣士林地方法院 115 年度 審簡 字第 701 號刑事判決（5K）  115.07.01  竊盜  
... 115年4 月 2日書記官林秀玉附錄本案所犯法條全文：中華民國刑法第320條意圖為自己或第三人不法之所有，而竊取他人之動產者，為竊盜罪，處 5 年以下有期徒刑、拘役或 50 萬元以下罰金。意圖